# Adversarial Data Protection Gradio Demo

Upload one or more images, choose a protection technique, and inspect the protected output, amplified perturbation map, image-quality metrics, and technique-specific model effect metrics. Resize mode is the default for Colab T4; patch mode is available as a slower advanced option.

In [ ]:
# Cell 1 - Setup (KHONG thay doi thu tu)
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ngocvuq4/adversarial-data-protection.git"
REPO_DIR = Path("/content/adversarial-data-protection")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    CACHE_DIR = Path("/content/drive/MyDrive/adversarial-data-protection/model_cache")
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")
    os.environ["XDG_CACHE_HOME"] = str(CACHE_DIR / "xdg")
    os.environ["CLIP_DOWNLOAD_ROOT"] = str(CACHE_DIR / "clip")
    Path(os.environ["CLIP_DOWNLOAD_ROOT"]).mkdir(parents=True, exist_ok=True)
    print("Model cache:", CACHE_DIR)

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print("Installing: requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import gc

import torch
import torch.nn.functional as F
import torchvision
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import base64
import io

import gradio as gr

from src.evaluation import compute_linf, compute_psnr, compute_ssim
from src.models import get_surrogate_resnet50
from src.pipeline import (
    image_to_tensor,
    image_to_tensor_no_resize,
    protect_patches,
    tensor_to_image,
)
from src.techniques.cloaking import cloak_images
from src.techniques.concept_poisoning import get_text_embedding, get_text_embeddings, load_clip_model, poison_images
from src.techniques.unlearnable import generate_unlearnable_batch

TECH_UNLEARNABLE = "Unlearnable Demo"
TECH_CLOAKING = "General Feature-space Cloaking"
TECH_CONCEPT = "CLIP-space Concept Poisoning Proxy"

surrogate = get_surrogate_resnet50(device)
clip_model = None
unlearnable_model = None

DEFAULT_CONCEPT_CANDIDATES = [
    "a photo of a dog",
    "a photo of a cat",
    "a photo of a car",
    "a photo of an airplane",
    "a photo of a flower",
    "a photo of a tree",
    "a photo of a chair",
    "a photo of a bird",
    "a photo of a face",
    "a photo of a building",
]

NOISE_DICT_PATH = "results/unlearnable_noise_dict.pt"
unlearnable_noise_dict = None
if os.path.exists(NOISE_DICT_PATH):
    unlearnable_noise_dict = torch.load(NOISE_DICT_PATH, map_location=device)
    print(f"[OK] Loaded {NOISE_DICT_PATH}")
else:
    print(
        f"[INFO] Missing {NOISE_DICT_PATH}. Unlearnable mode will use a fast mini-PGD fallback "
        "with ImageNet pseudo-labels instead of precomputed CIFAR class-wise noise."
    )


def _get_clip_model():
    global clip_model
    if clip_model is None:
        clip_model, _ = load_clip_model("ViT-B/32", device, download_root=os.environ.get("CLIP_DOWNLOAD_ROOT"))
    return clip_model


def _get_unlearnable_model():
    global unlearnable_model
    if unlearnable_model is None:
        weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
        backbone = torchvision.models.resnet18(weights=weights).to(device)
        backbone.eval()
        for param in backbone.parameters():
            param.requires_grad = False

        mean = torch.tensor(weights.transforms().mean, device=device).view(1, 3, 1, 1)
        std = torch.tensor(weights.transforms().std, device=device).view(1, 3, 1, 1)

        class ImageNetClassifier(torch.nn.Module):
            def __init__(self, model, mean, std):
                super().__init__()
                self.model = model
                self.register_buffer("mean", mean)
                self.register_buffer("std", std)

            def forward(self, x):
                return self.model((x - self.mean.to(dtype=x.dtype)) / self.std.to(dtype=x.dtype))

        unlearnable_model = ImageNetClassifier(backbone, mean, std).to(device).eval()
    return unlearnable_model


def _load_images(files):
    if not files:
        raise gr.Error("Please upload at least one image.")
    images = []
    for file_obj in files:
        file_path = getattr(file_obj, "name", file_obj)
        images.append(Image.open(file_path).convert("RGB"))
    return images


def _preview_image(image):
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    return Image.fromarray(np.asarray(image)).convert("RGB")


def _image_to_data_uri(image):
    pil_image = _preview_image(image)
    buffer = io.BytesIO()
    pil_image.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/png;base64,{encoded}"


def _image_panel(image, title):
    uri = _image_to_data_uri(image)
    return (
        '<div class="image-output-panel">'
        f'<div class="image-output-title">{title}</div>'
        f'<img src="{uri}" alt="{title}" />'
        '</div>'
    )


def resize_cifar_noise_for_ui(noise, target_hw, epsilon, source_epsilon=0.03):
    noise = noise.unsqueeze(0).float().to(device)
    noise = F.interpolate(noise, size=target_hw, mode="bilinear", align_corners=False)
    return noise * (float(epsilon) / source_epsilon)


def _protect_unlearnable_demo(batch, epsilon, pgd_steps=12):
    if unlearnable_noise_dict is not None:
        noise = resize_cifar_noise_for_ui(
            unlearnable_noise_dict[0],
            target_hw=batch.shape[-2:],
            epsilon=epsilon,
        ).to(batch.device)
        return torch.clamp(batch + noise, 0.0, 1.0)

    model = _get_unlearnable_model()
    with torch.no_grad():
        pseudo_y = model(batch.to(device)).argmax(dim=1)
    return generate_unlearnable_batch(
        model,
        batch.to(device),
        pseudo_y,
        epsilon=epsilon,
        pgd_steps=pgd_steps,
        pgd_alpha=max(epsilon / max(pgd_steps, 1), 1 / 255),
    )


def _protect_cloaking(batch, epsilon, pgd_steps):
    return cloak_images(
        surrogate,
        batch,
        epsilon=epsilon,
        pgd_steps=pgd_steps,
        pgd_alpha=max(epsilon / 10, 1 / 255),
        target_mode="max_dist",
        device=device,
    )


def _protect_concept_poisoning(batch, epsilon, pgd_steps, target_concept):
    model = _get_clip_model()
    return poison_images(
        model,
        batch,
        target_concept=target_concept,
        epsilon=epsilon,
        pgd_steps=pgd_steps,
        pgd_alpha=max(epsilon * 2 / pgd_steps, 1 / 255),
        device=device,
    )


def _noise_image(x_orig, x_protected, scale, original_size=None):
    noise_vis = ((x_protected.detach().cpu() - x_orig.detach().cpu()) * scale + 0.5).clamp(0, 1)
    return tensor_to_image(noise_vis, original_size=original_size)


def _metric_tensor(x, size=224):
    if x.shape[-2:] == (size, size):
        return x
    return F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)


def _cloaking_effect_metrics(x_orig, x_protected):
    x_orig_m = _metric_tensor(x_orig).to(device)
    x_prot_m = _metric_tensor(x_protected).to(device)
    surrogate.eval()
    with torch.no_grad():
        f_orig = F.normalize(surrogate(x_orig_m).float(), dim=1)
        f_prot = F.normalize(surrogate(x_prot_m).float(), dim=1)
        cosine = F.cosine_similarity(f_orig, f_prot, dim=1)
        l2_shift = torch.norm(f_prot - f_orig, p=2, dim=1)
    return {
        "feature_cosine_orig_protected": cosine.detach().cpu(),
        "feature_l2_shift": l2_shift.detach().cpu(),
    }


def _candidate_texts_for_ui(target_concept):
    target = target_concept.strip() if target_concept else "a photo of a dog"
    texts = [target]
    for candidate in DEFAULT_CONCEPT_CANDIDATES:
        if candidate.lower() != target.lower():
            texts.append(candidate)
    return texts


def _rank_of_target(similarities, target_index=0):
    order = torch.argsort(similarities, dim=1, descending=True)
    ranks = []
    for row in order:
        ranks.append((row == target_index).nonzero(as_tuple=False).item() + 1)
    return torch.tensor(ranks)


def _clip_similarity_metrics(x_orig, x_protected, target_concept):
    model = _get_clip_model()
    x_orig_m = _metric_tensor(x_orig).to(device)
    x_prot_m = _metric_tensor(x_protected).to(device)
    target_emb = get_text_embedding(model, target_concept, device)
    candidate_texts = _candidate_texts_for_ui(target_concept)
    candidate_emb = get_text_embeddings(model, candidate_texts, device)
    clip_mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
    clip_std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)
    with torch.no_grad():
        orig_emb = F.normalize(model.encode_image((x_orig_m - clip_mean) / clip_std).float(), dim=-1)
        prot_emb = F.normalize(model.encode_image((x_prot_m - clip_mean) / clip_std).float(), dim=-1)
        before = F.cosine_similarity(orig_emb, target_emb.expand_as(orig_emb), dim=1)
        after = F.cosine_similarity(prot_emb, target_emb.expand_as(prot_emb), dim=1)
        sims_before = orig_emb @ candidate_emb.T
        sims_after = prot_emb @ candidate_emb.T
        rank_before = _rank_of_target(sims_before, 0)
        rank_after = _rank_of_target(sims_after, 0)
        top_before_idx = sims_before.mean(dim=0).argmax().item()
        top_after_idx = sims_after.mean(dim=0).argmax().item()
        top_after = torch.topk(sims_after.mean(dim=0), k=min(5, len(candidate_texts)))
    return {
        "clip_target_similarity_before": before.detach().cpu(),
        "clip_target_similarity_after": after.detach().cpu(),
        "clip_target_similarity_delta": (after - before).detach().cpu(),
        "candidate_texts": candidate_texts,
        "target_rank_before": rank_before.detach().cpu(),
        "target_rank_after": rank_after.detach().cpu(),
        "target_rank_improvement": (rank_before - rank_after).detach().cpu(),
        "top_concept_before": candidate_texts[top_before_idx],
        "top_concept_after": candidate_texts[top_after_idx],
        "top_after_texts": [candidate_texts[i] for i in top_after.indices.detach().cpu().tolist()],
        "top_after_scores": top_after.values.detach().cpu(),
    }


def _quality_lines(x_orig, x_protected):
    return [
        "Image quality",
        f"  PSNR: {compute_psnr(x_orig, x_protected):.2f} dB",
        f"  SSIM: {compute_ssim(x_orig, x_protected):.4f}",
        f"  Linf: {compute_linf(x_orig, x_protected):.4f}",
    ]


def _effect_lines(technique, x_orig, x_protected, target_concept):
    lines = ["", "Model evidence"]
    if technique == TECH_CLOAKING:
        metrics = _cloaking_effect_metrics(x_orig, x_protected)
        cos = metrics["feature_cosine_orig_protected"]
        shift = metrics["feature_l2_shift"]
        lines.extend([
            f"  Feature cosine: {cos.mean().item():.4f}",
            f"  Feature L2 shift: {shift.mean().item():.4f}",
            "  Result: embedding shifted",
        ])
    elif technique == TECH_CONCEPT:
        metrics = _clip_similarity_metrics(x_orig, x_protected, target_concept)
        before = metrics["clip_target_similarity_before"]
        after = metrics["clip_target_similarity_after"]
        delta = metrics["clip_target_similarity_delta"]
        rank_before = metrics["target_rank_before"].float()
        rank_after = metrics["target_rank_after"].float()
        lines.extend([
            f"  Target: {target_concept}",
            f"  Similarity: {before.mean().item():.4f} -> {after.mean().item():.4f} ({delta.mean().item():+.4f})",
            f"  Target rank: #{rank_before.mean().item():.2f} -> #{rank_after.mean().item():.2f}",
            f"  Top concept after: {metrics['top_concept_after']}",
            "  Result: target concept moved up",
        ])
    else:
        lines.extend([
            "  Dataset-level method",
            "  UI shows perturbation quality only",
            "  Benchmark: 85.28% clean -> 10.37% protected",
        ])
    return lines


def _protect_resize_batch(images, technique, epsilon, target_concept, pgd_steps, noise_scale):
    original_sizes = [img.size for img in images]
    x_orig = torch.cat([image_to_tensor(img) for img in images], dim=0).to(device)

    if technique == TECH_UNLEARNABLE:
        x_protected = _protect_unlearnable_demo(x_orig, epsilon, pgd_steps)
    elif technique == TECH_CLOAKING:
        x_protected = _protect_cloaking(x_orig, epsilon, pgd_steps)
    elif technique == TECH_CONCEPT:
        x_protected = _protect_concept_poisoning(x_orig, epsilon, pgd_steps, target_concept)
    else:
        raise gr.Error(f"Unknown technique: {technique}")

    originals, protected, noises = [], [], []
    for idx, img in enumerate(images):
        originals.append(_preview_image(img))
        protected.append(_preview_image(tensor_to_image(x_protected[idx], original_size=original_sizes[idx])))
        noises.append(_preview_image(_noise_image(x_orig[idx], x_protected[idx], noise_scale, original_size=original_sizes[idx])))
    return originals, protected, noises, x_orig.detach().cpu(), x_protected.detach().cpu()


def _protect_patch_images(images, technique, epsilon, target_concept, pgd_steps, noise_scale):
    originals, protected, noises = [], [], []
    x_orig_all, x_prot_all = [], []

    if technique == TECH_UNLEARNABLE:
        technique_fn = lambda batch: _protect_unlearnable_demo(batch, epsilon, pgd_steps)
    elif technique == TECH_CLOAKING:
        technique_fn = lambda batch: _protect_cloaking(batch, epsilon, pgd_steps)
    elif technique == TECH_CONCEPT:
        technique_fn = lambda batch: _protect_concept_poisoning(batch, epsilon, pgd_steps, target_concept)
    else:
        raise gr.Error(f"Unknown technique: {technique}")

    for idx, img in enumerate(images):
        out_img, x_protected = protect_patches(
            img,
            technique_fn,
            batch_size=4,
            patch_size=224,
            device=device,
        )
        x_orig = image_to_tensor_no_resize(img).clamp(0, 1)
        originals.append(_preview_image(img))
        protected.append(_preview_image(out_img))
        noises.append(_preview_image(_noise_image(x_orig.squeeze(0), x_protected.squeeze(0), noise_scale)))
        x_orig_all.append(_metric_tensor(x_orig))
        x_prot_all.append(_metric_tensor(x_protected))

    return originals, protected, noises, torch.cat(x_orig_all, dim=0), torch.cat(x_prot_all, dim=0)


def protect_images(files, technique, epsilon, target_concept, pgd_steps, processing_mode, noise_scale):
    images = _load_images(files)
    epsilon = float(epsilon)
    pgd_steps = int(pgd_steps)
    noise_scale = float(noise_scale)
    concept = target_concept.strip() if target_concept else "a photo of a cat"
    processing_mode = processing_mode.lower()

    if processing_mode == "patch":
        originals, protected, noises, x_orig, x_protected = _protect_patch_images(
            images, technique, epsilon, concept, pgd_steps, noise_scale
        )
        mode_note = "Patch mode: each image is processed independently in 224x224 patch batches; this is slower but preserves more detail."
    else:
        originals, protected, noises, x_orig, x_protected = _protect_resize_batch(
            images, technique, epsilon, concept, pgd_steps, noise_scale
        )
        mode_note = "Resize mode: all uploaded images are batched at 224x224; General Cloaking can select max_dist targets across the uploaded batch."

    lines = [
        f"Technique: {technique}",
        f"Images: {len(images)}",
        f"Epsilon: {epsilon:.4f}",
        f"PGD steps: {pgd_steps}",
        mode_note,
        "",
        *(_quality_lines(x_orig, x_protected)),
        "",
        *(_effect_lines(technique, x_orig, x_protected, concept)),
    ]

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return _image_panel(originals[0], "Original preview"), _image_panel(protected[0], "Protected preview"), _image_panel(noises[0], "Noise map preview"), "\n".join(lines)


APP_CSS = """
body, .gradio-container {
  background-color: #eef2f7 !important;
  background-image:
    repeating-linear-gradient(0deg, rgba(15, 23, 42, 0.045) 0, rgba(15, 23, 42, 0.045) 1px, transparent 1px, transparent 28px),
    repeating-linear-gradient(90deg, rgba(15, 23, 42, 0.045) 0, rgba(15, 23, 42, 0.045) 1px, transparent 1px, transparent 28px);
}
.gradio-container { max-width: 1280px !important; margin: 0 auto; padding-top: 18px !important; }
#hero {
  padding: 20px 24px;
  border: 1px solid #cbd5e1;
  border-radius: 8px;
  background: #111827;
  color: #f8fafc;
  box-shadow: 0 18px 42px rgba(15, 23, 42, 0.18);
}
#hero h1 { margin: 0 0 6px 0; font-size: 30px; line-height: 1.15; letter-spacing: 0; }
#hero p { margin: 0; color: #cbd5e1; font-size: 15px; }
#control-panel, #metric-panel {
  border: 1px solid #d6dee8;
  border-radius: 8px;
  padding: 14px;
  background: rgba(255, 255, 255, 0.96);
  box-shadow: 0 12px 28px rgba(15, 23, 42, 0.10);
}
#run-button button { width: 100%; min-height: 44px; font-weight: 700; }
#metrics-box textarea { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 13px; line-height: 1.45; }
.gallery-panel {
  border: 1px solid #d6dee8;
  border-radius: 8px;
  padding: 8px;
  background: rgba(255, 255, 255, 0.96);
  box-shadow: 0 10px 24px rgba(15, 23, 42, 0.08);
}
.image-output-panel {
  min-height: 300px;
  display: flex;
  flex-direction: column;
  gap: 8px;
}
.image-output-title {
  font-weight: 700;
  font-size: 13px;
  color: #111827;
}
.image-output-panel img {
  width: 100%;
  height: 270px;
  object-fit: contain;
  border-radius: 6px;
  background: #111827;
}
"""

with gr.Blocks(title="Adversarial Data Protection Demo", css=APP_CSS) as demo:
    gr.HTML(
        """
        <section id="hero">
          <h1>Adversarial Data Protection Demo</h1>
          <p>PGD-based image protection with Unlearnable perturbations, feature-space cloaking, and CLIP-space concept poisoning.</p>
        </section>
        """
    )

    with gr.Row(equal_height=False):
        with gr.Column(scale=4, elem_id="control-panel"):
            gr.Markdown("### Configuration")
            input_files = gr.Files(
                label="Images",
                file_count="multiple",
                file_types=["image"],
            )
            technique = gr.Radio(
                [TECH_CLOAKING, TECH_CONCEPT, TECH_UNLEARNABLE],
                label="Technique",
                value=TECH_CLOAKING,
            )
            target_concept = gr.Textbox(
                label="Target concept",
                value="a photo of a dog",
                placeholder="a photo of a dog",
            )
            with gr.Row():
                epsilon = gr.Slider(minimum=0.01, maximum=0.1, step=0.01, value=0.03, label="Epsilon")
                pgd_steps = gr.Slider(minimum=5, maximum=20, step=1, value=10, label="PGD steps")
            with gr.Accordion("Advanced", open=False):
                processing_mode = gr.Radio(["resize", "patch"], label="Processing mode", value="resize")
                noise_scale = gr.Slider(minimum=5, maximum=30, step=5, value=10, label="Noise map scale")
            run_btn = gr.Button("Protect Images", variant="primary", elem_id="run-button")

        with gr.Column(scale=6, elem_id="metric-panel"):
            gr.Markdown("### Model Evidence")
            metrics = gr.Textbox(
                label="Metrics",
                lines=10,
                elem_id="metrics-box",
                show_copy_button=True,
            )

    gr.Markdown("### Visual Outputs")
    with gr.Row():
        with gr.Column(elem_classes=["gallery-panel"]):
            original_preview = gr.HTML(label="Original preview")
        with gr.Column(elem_classes=["gallery-panel"]):
            protected_preview = gr.HTML(label="Protected preview")
        with gr.Column(elem_classes=["gallery-panel"]):
            noise_preview = gr.HTML(label="Noise map preview")

    run_btn.click(
        fn=protect_images,
        inputs=[input_files, technique, epsilon, target_concept, pgd_steps, processing_mode, noise_scale],
        outputs=[original_preview, protected_preview, noise_preview, metrics],
        show_progress="full",
    )

try:
    demo.queue(concurrency_count=1).launch(share=True)
except TypeError:
    # Newer Gradio versions removed concurrency_count from Blocks.queue().
    demo.queue(default_concurrency_limit=1).launch(share=True)
